# 2×2 CUDA ablation: combined Hessian × matrix exponential

This notebook measures two independent exact-Hessian optimizations for issue #122 in one Colab session:

| | scaling-and-squaring (`ss`) | native `torch.linalg.matrix_exp` |
|---|---|---|
| legacy Hessian | baseline | native expm only |
| combined Hessian | combined transform only | both optimizations |

The quick test captures every real IPOPT Hessian callback on a one-day problem, evaluates closures at identical inputs, checks numerical agreement, and times repeated calls. The final cell optionally runs all four arms on the full five-day problem and reports solver health plus complete per-sensor RMSE tables.

The native backend is experimental; `ss` remains the production default until value, derivative, stability, and CUDA timing results pass. CUDA graphs stay disabled and Hessian chunking stays fixed.

Use **Runtime → Change runtime type → GPU**, then **Run all**. An A100 is preferred because this workload uses FP64.

In [ ]:
# Quieten the translator's per-type namespace warnings. They are a known
# issue (#114), and several model builds would otherwise bury the A/B results.
import warnings

warnings.filterwarnings("ignore", message="Failed to parse namespace")
warnings.filterwarnings("ignore", message="Failed to parse ontology namespace")
warnings.filterwarnings("ignore", message='Neither "df", "filename", nor "uuid"')
# --- Setup (Colab-aware) ---------------------------------------------------
TWIN4BUILD_REF = "feature/issue-122/cuda-graph-hessian"

try:
    import twin4build as tb
except ImportError:
    # subprocess rather than %pip so the ref interpolates unambiguously.
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         f"git+https://github.com/JBjoernskov/Twin4Build.git@{TWIN4BUILD_REF}"],
        check=True,
    )
    import twin4build as tb

# Fail loudly now rather than after the benchmark has run.
import inspect

import twin4build.estimator._transcription as _tr
import twin4build.systems.utils.discrete_statespace_system as _dss

_tr_src = inspect.getsource(_tr)
_dss_src = inspect.getsource(_dss)
_missing = [n for n in ("TWIN4BUILD_COMBINED_HESSIAN", "exact_hessian",
                        "boundary_state_init") if n not in _tr_src]
if "TWIN4BUILD_MATRIX_EXP" not in _dss_src:
    _missing.append("TWIN4BUILD_MATRIX_EXP")
if _missing:
    # Reaching here after a forced reinstall means the OLD module is still
    # loaded in this session -- pip cannot replace an already-imported module.
    raise RuntimeError(
        "The installed twin4build is missing: " + ", ".join(_missing)
        + f"""
Installed at: {tb.__file__}
Install a ref that has these, then restart the runtime:
    pip install -q --force-reinstall --no-deps git+https://github.com/JBjoernskov/Twin4Build.git@{TWIN4BUILD_REF}
    (Colab: Runtime > DISCONNECT AND DELETE RUNTIME, then Run all.
     "Restart session" alone is NOT enough -- the stale package survives it.)"""
    )

import functools
import os
import platform
import time

import numpy as np
import pandas as pd
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU required: Runtime > Change runtime type > GPU")

props = torch.cuda.get_device_properties(0)
print("GPU:", props.name, f"({props.total_memory / 1e9:.1f} GB, sm_{props.major}{props.minor})")
print("CPU:", platform.processor() or platform.machine(), "cores:", os.cpu_count())
print("Torch:", torch.__version__)
print("Twin4Build:", tb.__file__)
print("Ref:", TWIN4BUILD_REF)

# CUDA graphs are deliberately disabled: this notebook measures eager Hessian
# organization, and failed graph capture would poison the CUDA context.
os.environ["TWIN4BUILD_CUDA_GRAPH"] = "0"
os.environ["TWIN4BUILD_HESS_CHUNK_DIV"] = "1"
os.environ["TWIN4BUILD_MATRIX_EXP"] = "ss"

In [ ]:
import datetime
import importlib.util as _ilu
import pathlib

from dateutil import tz

import twin4build.examples as _ex_pkg
import twin4build.examples.utils as utils

# The data package shadows full_workflow_example.py, so load the module by path.
_path = pathlib.Path(_ex_pkg.__file__).parent / "full_workflow_example.py"
_spec = _ilu.spec_from_file_location("_fwe_combined_hessian", _path)
_mod = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
fcn = _mod.fcn

STEP = 1200
N_WARMUP = 20
START = [datetime.datetime(2023, 12, 2, tzinfo=tz.gettz("Europe/Copenhagen"))]
QUICK_END = [datetime.datetime(2023, 12, 3, tzinfo=tz.gettz("Europe/Copenhagen"))]
FULL_END = [datetime.datetime(2023, 12, 7, tzinfo=tz.gettz("Europe/Copenhagen"))]


def build_model(tag):
    model = tb.Model(id=tag)
    model.load(
        semantic_model_filename=utils.get_path(
            ["estimator_example", "one_room_example_model.xlsm"]
        ),
        fcn=fcn,
    )
    model.to("cuda", torch.float64)
    return model


def build_parameters(model):
    c = model.components
    space, heater = c["office"], c["office_space_heater"]
    hc, cc = c["office_temperature_heating_controller"], c["office_co2_controller"]
    valve = c["office_space_heater_valve"]
    sup, exh = c["office_supply_damper"], c["office_exhaust_damper"]
    occ, wall = c["office_occupancy"], c["office_boundary_wall"]
    det = c["office_occupancy_detector"]
    return [
        (space, "thermal.C_air", 5e5, 1e4, 5e5),
        (space, "thermal.C_wall", 1e6, 1e5, 3e6),
        (wall, "C", 1e6, 1e4, 1e7),
        (space, "thermal.R_out", 0.5, 0.01, 1),
        (space, "thermal.R_in", 0.1, 0.01, 1),
        (wall, "R_a", 0.04, 1e-4, 1),
        (wall, "R_b", 0.04, 1e-4, 1),
        (space, "thermal.f_wall", 0.1, 0, 10),
        (space, "thermal.f_air", 0.1, 0, 10),
        (space, "thermal.Q_occ_gain", 100.0, 10, 200),
        (heater, "thermalMassHeatCapacity", 1e4, 1e3, 2e5),
        (heater, "UA", None, 1, 100),
        (hc, "kp", 0.005, 1e-5, 1, "private"),
        (cc, "kp", 0.0001, 1e-5, 1, "private"),
        ([hc, cc], "Ti", 30, 1, 300, "private"),
        ([hc, cc], "Td", 0, 0, 1, "private"),
        (valve, "waterFlowRateMax", 0.001, 1e-6, 0.1),
        (valve, "valveAuthority", 1, 0.4, 1),
        ([sup, occ.supply_damper], "a", 1, 1, 10, "shared"),
        ([sup, occ.supply_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([exh, occ.exhaust_damper], "a", 1, 1, 10, "shared"),
        ([exh, occ.exhaust_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([space, occ], "mass.V", 65, 50, 80, "shared"),
        ([space, occ], "mass.G_occ", 1e-6, 1e-6, 1e-5, "shared"),
        ([space, occ], "mass.m_inf", 0.001, 1e-4, 0.01, "shared"),
        (det, "threshold", 1.0, 0.02, 5.0),
    ]


def build_measurements(model):
    c = model.components
    return [
        (c["office_valve_position_sensor"], 0.05 / 2),
        (c["office_temperature_sensor"], 0.1 / 2),
        (c["office_damper_position_sensor"], 0.05 / 2),
        (c["office_co2_sensor"], 30 / 2),
    ]


BASE_OPTIONS = {
    "boundary_state_init": "rollout",
    "early_stopping": False,
    "exact_hessian": True,
}
print("Model builders ready")

In [ ]:
import twin4build.estimator._casadi_ipopt as _ipopt

ARMS = [
    ("legacy_ss", False, "ss"),
    ("combined_ss", True, "ss"),
    ("legacy_native", False, "native"),
    ("combined_native", True, "native"),
]
CAPTURED = {}


def apply_arm_env(combined, matrix_exp):
    os.environ["TWIN4BUILD_COMBINED_HESSIAN"] = "1" if combined else "0"
    os.environ["TWIN4BUILD_MATRIX_EXP"] = matrix_exp


def capture_arm(label, combined, matrix_exp, maxiter=2):
    """Build the real callback and retain its first real IPOPT arguments."""
    apply_arm_env(combined, matrix_exp)
    original = _ipopt.solve_ipopt_constrained

    def wrapped(
        x0, lb, ub, fun, grad, n_g, g_fun, g_jac_vals, jr, jc,
        options=None, *, hess_vals=None, **kwargs
    ):
        @functools.wraps(hess_vals)
        def record(*args):
            if "args" not in CAPTURED[label]:
                CAPTURED[label]["args"] = tuple(
                    np.asarray(a).copy() if hasattr(a, "__len__") else a
                    for a in args
                )
            return hess_vals(*args)

        CAPTURED[label]["fn"] = hess_vals
        return original(
            x0, lb, ub, fun, grad, n_g, g_fun, g_jac_vals, jr, jc, options,
            hess_vals=record, **kwargs
        )

    CAPTURED[label] = {
        "combined": combined,
        "matrix_exp": matrix_exp,
    }
    _ipopt.solve_ipopt_constrained = wrapped
    try:
        model = build_model(f"capture_{label}")
        estimator = tb.Estimator(tb.Simulator(model))
        options = dict(BASE_OPTIONS, maxiter=maxiter)
        estimator.estimate(
            START, QUICK_END, STEP,
            build_parameters(model), build_measurements(model),
            n_warmup=N_WARMUP,
            method=("casadi", "ipopt", "ad", "collocation"),
            options=options,
        )
    finally:
        _ipopt.solve_ipopt_constrained = original


for label, combined, matrix_exp in ARMS:
    print(f"Capturing {label} callback ...", flush=True)
    try:
        capture_arm(label, combined, matrix_exp)
    except Exception as exc:
        CAPTURED.setdefault(label, {})["error"] = (
            f"{type(exc).__name__}: {str(exc).splitlines()[0]}"
        )
        print("  FAILED:", CAPTURED[label]["error"])
    finally:
        torch.cuda.empty_cache()

available = [label for label, _, _ in ARMS if "fn" in CAPTURED.get(label, {})]
if "legacy_ss" not in available:
    raise RuntimeError("The scaling-and-squaring baseline failed; ablation is invalid")

# Evaluate every available closure at exactly the same primal point, objective
# scale, and constraint multipliers. This is stronger than endpoint comparison.
common_args = CAPTURED["legacy_ss"]["args"]
outputs = {}
rows = []
for label, combined, matrix_exp in ARMS:
    if label not in available:
        rows.append({
            "arm": label,
            "combined": combined,
            "matrix_exp": matrix_exp,
            "error": CAPTURED.get(label, {}).get("error", "not captured"),
        })
        continue
    fn = CAPTURED[label]["fn"]
    outputs[label] = fn(*common_args)
    torch.cuda.synchronize()
    samples = []
    for _ in range(3):
        started = time.perf_counter()
        fn(*common_args)
        torch.cuda.synchronize()
        samples.append(time.perf_counter() - started)
    rows.append({
        "arm": label,
        "combined": combined,
        "matrix_exp": matrix_exp,
        "median_callback_s": float(np.median(samples)),
        "min_callback_s": float(np.min(samples)),
        "samples_s": [round(x, 4) for x in samples],
        "error": None,
    })

comparison_rows = []
comparison_pairs = [
    ("legacy_ss", "combined_ss", "Hessian combination with ss"),
    ("legacy_native", "combined_native", "Hessian combination with native"),
    ("legacy_ss", "legacy_native", "Matrix-exp backend with legacy Hessian"),
    ("combined_ss", "combined_native", "Matrix-exp backend with combined Hessian"),
]
for reference, candidate, comparison in comparison_pairs:
    if reference not in outputs or candidate not in outputs:
        continue
    delta = np.abs(outputs[candidate] - outputs[reference])
    scale = max(float(np.max(np.abs(outputs[reference]))), 1e-12)
    comparison_rows.append({
        "comparison": comparison,
        "reference": reference,
        "candidate": candidate,
        "max_abs_delta": float(delta.max()),
        "relative_max_delta": float(delta.max() / scale),
        "allclose_1e-9": bool(np.allclose(
            outputs[candidate], outputs[reference], rtol=1e-9, atol=1e-10
        )),
        "all_finite": bool(np.isfinite(outputs[candidate]).all()),
    })

callback_results = pd.DataFrame(rows)
baseline = callback_results.loc[
    callback_results.arm == "legacy_ss", "median_callback_s"
].iloc[0]
callback_results["speedup_vs_legacy_ss"] = (
    baseline / callback_results["median_callback_s"]
)
callback_ablation = callback_results.pivot(
    index="combined", columns="matrix_exp", values="median_callback_s"
)
comparison_results = pd.DataFrame(comparison_rows)

with pd.option_context("display.max_columns", None, "display.width", 200):
    print("Callback timing details")
    display(callback_results)
    print("2×2 median callback seconds")
    display(callback_ablation)
    print("Same-input numerical comparisons")
    display(comparison_results)

## Optional end-to-end 2×2 ablation

The callback comparison above isolates both optimizations. The next cell measures whether they survive IPOPT overhead on the full five-day problem. It runs up to 100 iterations for each of four arms and can take 20–45 minutes on an A100; native expm may be much slower if PyTorch falls back to sequential batching.

Endpoint fits can differ because tiny floating-point changes alter IPOPT's trajectory. Treat same-input callback comparisons as the numerical check, then use iterations, status, feasibility, objective, and the complete RMSE table to judge end-to-end behavior. A failed experimental native arm is recorded rather than aborting the remaining grid.

In [ ]:
RUN_END_TO_END = True
END_TO_END_ITERS = 100

end_to_end_rows = []
rmse_rows = []
if RUN_END_TO_END:
    for label, combined_flag, matrix_exp in ARMS:
        print(f"\nRunning full-horizon {label} arm ...", flush=True)
        apply_arm_env(combined_flag, matrix_exp)
        model = estimator = None
        started = time.perf_counter()
        try:
            model = build_model(f"e2e_{label}")
            estimator = tb.Estimator(tb.Simulator(model))
            options = dict(BASE_OPTIONS, maxiter=END_TO_END_ITERS)
            result = estimator.estimate(
                START, FULL_END, STEP,
                build_parameters(model), build_measurements(model),
                n_warmup=N_WARMUP,
                method=("casadi", "ipopt", "ad", "collocation"),
                options=options,
            )
            elapsed = time.perf_counter() - started
            audit = result.get("transcription_audit", {})
            per_sensor = audit.get("per_sensor", {})
            end_to_end_rows.append({
                "arm": label,
                "combined": combined_flag,
                "matrix_exp": matrix_exp,
                "seconds": elapsed,
                "iterations": result.get("iterations"),
                "final_objective": result.get("final_objective"),
                "success": result.get("success"),
                "return_status": audit.get(
                    "return_status", result.get("message")
                ),
                "max_defect": audit.get("max_defect", np.nan),
                "error": None,
            })
            for sensor, values in per_sensor.items():
                rmse_rows.append({
                    "arm": label,
                    "combined": combined_flag,
                    "matrix_exp": matrix_exp,
                    "sensor": sensor,
                    "nlp_rmse": float(values["nlp_rmse"]),
                    "rollout_rmse": float(values["rollout_rmse"]),
                    "do_step_rmse": float(values["do_step_rmse"]),
                })
        except Exception as exc:
            end_to_end_rows.append({
                "arm": label,
                "combined": combined_flag,
                "matrix_exp": matrix_exp,
                "seconds": time.perf_counter() - started,
                "iterations": np.nan,
                "final_objective": np.nan,
                "success": False,
                "return_status": "FAILED",
                "max_defect": np.nan,
                "error": f"{type(exc).__name__}: {str(exc).splitlines()[0]}",
            })
            print("  FAILED:", end_to_end_rows[-1]["error"])
        finally:
            del estimator, model
            torch.cuda.empty_cache()

    end_to_end_results = pd.DataFrame(end_to_end_rows)
    baseline = end_to_end_results.loc[
        end_to_end_results.arm == "legacy_ss", "seconds"
    ].iloc[0]
    end_to_end_results["speedup_vs_legacy_ss"] = (
        baseline / end_to_end_results["seconds"]
    )
    end_to_end_ablation = end_to_end_results.pivot(
        index="combined", columns="matrix_exp", values="seconds"
    )
    rmse_results = pd.DataFrame(rmse_rows)
    rmse_comparison = (
        rmse_results.pivot(
            index="sensor",
            columns="arm",
            values=["nlp_rmse", "rollout_rmse", "do_step_rmse"],
        )
        if len(rmse_results)
        else pd.DataFrame()
    )

    with pd.option_context("display.max_columns", None, "display.width", 240):
        print("Solve summary")
        display(end_to_end_results)
        print("2×2 end-to-end seconds")
        display(end_to_end_ablation)
        print("Per-sensor RMSE comparison (raw sensor units)")
        display(rmse_comparison)
else:
    print("Skipped. Set RUN_END_TO_END=True and rerun this cell when desired.")